In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359

maxval=1e9
minval=1e-9



2025-06-23 15:25:42.499123: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-23 15:25:42.499184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-23 15:25:42.500059: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 15:25:42.506587: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-23 15:25:43.522153: W tensorflow/compiler/tf2

In [2]:
# os.chdir('SmartPix/data_generator')
os.chdir('/home/das214/SmartPix/mlp_enc_dev')
!pwd

/home/das214/SmartPix/mlp_enc_dev


In [3]:
from DG.OptimizedDataGenerator_v2 import OptimizedDataGenerator
from losses.diag_loss_nll import custom_diag_loss
from models.mlp_encoder_model import CreateModel

In [4]:
dataset_base_dir = "/depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train")
dataset_test_dir = os.path.join(dataset_base_dir, "test")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val")

batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_test_dir))

In [5]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_test_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,

#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )

# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# # training generator
# start_time = time.time()
# training_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_train_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = batch_size,
#     # optimize_batch_size = True,
#     file_count = train_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, # True 

#     tfrecords_dir = tfrecords_dir_train,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Training generator %s seconds ---" % (time.time() - start_time))

In [6]:
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=True,
)


Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json


In [7]:
diag_model=CreateModel(shape = (16,16,2), output = 8)
diag_model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3,  clipnorm=1.0),
    loss=custom_diag_loss,
    run_eagerly=True,
)

diag_model.summary()

2025-06-23 15:25:48.017787: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38397 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 7g.40gb, pci bus id: 0000:21:00.0, compute capability: 8.0
2025-06-23 15:25:48.316079: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls/ (InputLayer)    [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['input_pxls/[0][0]']         
 Pooling2D)                                                                                       
                                                                                                  
 average_pooling2d_1 (Avera  (None, 1, 16, 2)             0         ['input_pxls/[0][0]']         
 gePooling2D)                                                                                     
                                                                                 

In [8]:
from datetime import datetime

fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [9]:
print(fingerprint)

7fe8f1b2


In [10]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback

early_stopping_patience = 50

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort()
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [11]:
training_generator.__len__()

84

In [12]:
for i in range(training_generator.__len__()):

    X_batch, y_batch = training_generator[i]

    print("--- Running a single forward pass... ---")
    try:
        # Get the model's raw predictions
        predictions = diag_model(X_batch, training=True)

        # Use TensorFlow's built-in checker
        tf.debugging.check_numerics(predictions, "Model predictions contain NaN or Inf!")

        print("✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).")
        print("\nSample of predictions (first 5):")
        print(predictions.numpy()[:5])

    except Exception as e:
        print(f"❌ FAILURE: The NaN is being generated inside the model's forward pass.")
        print(f"Error: {e}")


--- Running a single forward pass... ---


2025-06-23 15:25:51.621344: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).

Sample of predictions (first 5):
[[ 0.06652832  0.04827881  0.6789551  -0.01580811 -0.36572266 -0.28302002
   0.42022705  0.11676025]
 [ 0.24127197  0.18341064  0.31115723 -0.08453369 -0.23883057 -0.22314453
   0.18560791  0.07769775]
 [ 0.07055664 -0.02130127  0.6816406  -0.00439453 -0.50231934 -0.2642212
   0.44104004  0.1786499 ]
 [ 0.17578125  0.02557373 -0.03460693  0.10229492 -0.03552246 -0.35931396
   0.10803223 -0.06152344]
 [ 0.44561768  0.40356445 -0.19262695 -0.36242676  0.12426758 -0.16680908
   0.3258667  -0.1697998 ]]
--- Running a single forward pass... ---
✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).

Sample of predictions (first 5):
[[ 0.07800293  0.13482666  0.36547852 -0.12268066 -0.22235107 -0.12597656
   0.31121826 -0.01617432]
 [ 0.02813721 -0.01751709  0.4156494   0.15887451 -0.43432617 -0.2008667
   0.36395264  0.13311768]
 [ 0.13018799  0.13085938 -0.224609

In [13]:
history = diag_model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=1000,
        shuffle=False,
        verbose=1
    )

Epoch 1/1000


2025-06-23 15:26:00.941654: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x5559443eb8b0
2025-06-23 15:26:01.604480: I external/local_xla/xla/service/service.cc:168] XLA service 0x555950104c10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-23 15:26:01.604535: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 7g.40gb, Compute Capability 8.0
2025-06-23 15:26:01.615722: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1750685161.731595  324058 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


77/84 [==========================>...] - ETA: 0s - loss: nan

2025-06-23 15:26:12.242909: W tensorflow/core/framework/op_kernel.cc:1827] UNKNOWN: InvalidArgumentError: {{function_node __wrapped__Reshape_device_/job:localhost/replica:0/task:0/device:GPU:0}} Input to reshape is a tensor with 2560000 values, but the requested shape has 20000 [Op:Reshape]
Traceback (most recent call last):

  File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow/python/ops/script_ops.py", line 270, in __call__
    ret = func(*args)

  File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "/depot/cms/kernels/python3/lib/python3.10/site-packages/keras/src/engine/data_adapter.py", line 917, in wrapped_generator
    for data in generator_f

InvalidArgumentError: in user code:

    File "/home/das214/SmartPix/mlp_enc_dev/losses/diag_loss_nll.py", line 27, in custom_diag_loss  *
        NLL = -dist.log_prob(y)
    File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow_probability/python/distributions/distribution.py", line 1287, in log_prob  **
        return self._call_log_prob(value, name, **kwargs)
    File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow_probability/python/distributions/distribution.py", line 1269, in _call_log_prob
        return self._log_prob(value, **kwargs)
    File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow_probability/python/internal/distribution_util.py", line 1350, in _fn
        return fn(*args, **kwargs)
    File "/depot/cms/kernels/python3/lib/python3.10/site-packages/tensorflow_probability/python/distributions/mvn_linear_operator.py", line 243, in _log_prob
        return super(MultivariateNormalLinearOperator, self)._log_prob(x)

In [ ]:
X_batch, y_batch = training_generator[0]

print("--- Running a single forward pass... ---")
try:
    # Get the model's raw predictions
    predictions = diag_model(X_batch, training=True)

    # Use TensorFlow's built-in checker
    tf.debugging.check_numerics(predictions, "Model predictions contain NaN or Inf!")

    print("✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).")
    print("\nSample of predictions (first 5):")
    print(predictions.numpy()[:5])

except Exception as e:
    print(f"❌ FAILURE: The NaN is being generated inside the model's forward pass.")
    print(f"Error: {e}")


--- Running a single forward pass... ---
❌ FAILURE: The NaN is being generated inside the model's forward pass.
Error: {{function_node __wrapped__CheckNumerics_device_/job:localhost/replica:0/task:0/device:GPU:0}} Model predictions contain NaN or Inf! : Tensor had NaN values [Op:CheckNumerics] name: 


2025-06-23 15:25:01.823355: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7f44dfe00000 = {1, 0} Model predictions contain NaN or Inf!
